# Classificação de doença cardíaca com a API Keras

Este notebook implementa uma solução completa de deep learning para estimar a probabilidade de doença cardíaca a partir do dataset `heart.csv`.

A solução inclui pré-processamento de variáveis numéricas e categóricas com camadas Keras, treinamento, avaliação e inferência para um novo paciente.

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "1")
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")
os.environ.setdefault("TF_DATA_PRIVATE_THREADPOOL_SIZE", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from tensorflow import keras

from ic.config import HeartConfig, PathsConfig
from ic.datasets.heart import load_heart_data, split_heart_data
from ic.modeling.heart import build_heart_classifier, make_heart_inputs
from ic.modeling.predict import predict_heart_patient
from ic.modeling.train import train_heart_classifier

keras.utils.set_random_seed(1337)
config = HeartConfig()
paths = PathsConfig()
print("Keras:", keras.__version__)

## 1. Carregamento e inspeção dos dados

O alvo `target` vale 0 para ausência e 1 para presença de doença cardíaca.

In [ ]:
dataframe = load_heart_data(config)
assert "target" in dataframe.columns
print("Shape:", dataframe.shape)
display(dataframe.head())

In [ ]:
heart_data = split_heart_data(dataframe, config)
train_dataframe = heart_data.train
val_dataframe = heart_data.validation

y_train = train_dataframe["target"].to_numpy(dtype="float32").reshape(-1, 1)
y_val = val_dataframe["target"].to_numpy(dtype="float32").reshape(-1, 1)

print(f"Treino: {len(train_dataframe)} amostras")
print(f"Validação: {len(val_dataframe)} amostras")
print("Distribuição no treino:", np.bincount(y_train.astype("int32").ravel()))

## 2. Preparação das entradas

As variáveis numéricas serão normalizadas por camadas `Normalization`. As variáveis categóricas serão convertidas em índices e codificadas com `IntegerLookup(output_mode="one_hot")`. O vocabulário é aprendido somente no conjunto de treino.

In [ ]:
model, categorical_vocabularies = build_heart_classifier(train_dataframe, config)
train_inputs = make_heart_inputs(train_dataframe, categorical_vocabularies)
val_inputs = make_heart_inputs(val_dataframe, categorical_vocabularies)
print({name: value.shape for name, value in train_inputs.items()})

## 3. Construção do modelo Keras

Será usado um modelo funcional com uma entrada nomeada para cada característica. A saída sigmoid representa a probabilidade estimada de doença cardíaca.

In [ ]:
model.summary()

## 4. Treinamento

O `EarlyStopping` interrompe o treinamento quando a loss de validação deixa de melhorar e restaura os melhores pesos.

In [ ]:
checkpoint_path = paths.checkpoints_dir / config.model_name
history = train_heart_classifier(
    model,
    train_dataframe,
    val_dataframe,
    categorical_vocabularies,
    config,
    checkpoint_path,
)

## 5. Avaliação e decisão

A rede produz uma probabilidade. Para transformar essa probabilidade em classe, usamos o limiar `0.5`.

In [ ]:
results = model.evaluate(val_inputs, y_val, return_dict=True, verbose=0)
print({name: round(float(value), 4) for name, value in results.items()})

probabilities = model.predict(val_inputs, verbose=0).ravel()
threshold = config.threshold
predicted_classes = (probabilities >= threshold).astype("int32")
true_classes = y_val.ravel().astype("int32")

print("Predições calculadas:", len(predicted_classes))

In [ ]:
comparison = pd.DataFrame(
    {
        "real": true_classes,
        "probabilidade": probabilities,
        "classe": predicted_classes,
    }
)
comparison.head(10)

## 6. Inferência para um novo paciente

A função abaixo aplica exatamente o mesmo mapeamento categórico usado no treino e retorna a probabilidade prevista.

In [ ]:
sample = {
    "age": 60,
    "sex": 1,
    "cp": 1,
    "trestbps": 145,
    "chol": 233,
    "fbs": 1,
    "restecg": 2,
    "thalach": 150,
    "exang": 0,
    "oldpeak": 2.3,
    "slope": 3,
    "ca": 0,
    "thal": "fixed",
}

sample_probability = predict_heart_patient(
    model,
    sample,
    categorical_vocabularies,
)
sample_class = int(sample_probability >= config.threshold)
print(f"Probabilidade estimada: {sample_probability:.3f}")
print(f"Classe prevista: {sample_class}")

## Conclusão

A solução usa um modelo Keras funcional para receber as 13 características, normalizar valores numéricos, codificar categorias, aprender uma representação não linear e gerar uma probabilidade binária. A qualidade final deve ser analisada pela métrica de validação, especialmente AUC, além da acurácia.